# LED-off baseline reference diagnostic

Use a random/forced/external-trigger run acquired with the LED off and the same PMT, HV, channel, bandwidth, termination, sampling, and voltage scale as the measurement data. Process one voltage at a time: the notebook builds a robust baseline template, diagnoses coherent pickup and residual noise, flags likely dark-pulse contamination, and saves a voltage-named `.npz` artifact.

The pointwise median—not the ordinary mean—is the primary template because occasional asynchronous dark pulses should not bias it.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter1d
from scipy.signal import welch

from lab_tools.io import iter_keysight_chunks
from pmt.io import extract_pmt_voltage

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True, "grid.alpha": 0.35})

## Configuration

In [ ]:
# Put the uploaded reference files here, or change this path.
reference_data_dir = Path("PMT_Data/Baseline_Reference")
# Select exactly one PMT voltage per run, for example '*_800V_*.h5'.
reference_glob = "*_800V_*.h5"
channel = "Channel 3"

# Start with a bounded sample for diagnostics; use None for every event once checked.
max_files = None
max_events = 20_000
chunk_size = 512
random_seed = 12345

# A real PMT pulse is broad/unipolar compared with high-frequency bipolar pickup.
# Smoothing is used only to flag contaminated reference traces, never to form
# the saved raw-sample noise distribution.
pulse_test_smoothing_ns = 1.0
pulse_test_snr = 8.0
template_iterations = 2
minimum_clean_fraction = 0.5

# Candidate maximum accidental hardware-trigger rates. Recommendations with
# fewer than minimum_expected_exceedances expected in this reference exposure
# are reported as statistically unsupported rather than silently extrapolated.
target_false_trigger_rates_hz = [100, 1_000, 10_000, 100_000]
minimum_expected_exceedances = 10
trigger_safety_margin_mV = 0.5

# Keep separate collections for bandwidth/filter/termination configurations.
# Batch_analysis.ipynb points each dataset at one of these directories.
reference_collection = "dark_counts"
output_root = Path("baseline_reference")

## Load raw reference waveforms

In [ ]:
reference_files = sorted(reference_data_dir.glob(reference_glob))
if max_files is not None:
    reference_files = reference_files[:max_files]
if not reference_files:
    raise FileNotFoundError(
        f"No reference files matched {reference_data_dir / reference_glob}. "
        "Upload the LED-off run or update reference_data_dir/reference_glob."
    )
reference_voltages_V = {extract_pmt_voltage(path) for path in reference_files}
if len(reference_voltages_V) != 1:
    raise ValueError(
        f"Reference files must contain exactly one voltage; found "
        f"{sorted(reference_voltages_V)}. Narrow reference_glob."
    )
reference_voltage_V = reference_voltages_V.pop()
output_dir = output_root / reference_collection
template_output = output_dir / f"{reference_voltage_V:g}V.npz"
print(f"Reference voltage: {reference_voltage_V:g} V")
print(f"Artifact collection: {output_dir}")

waveform_parts = []
time_ns = None
events_loaded = 0
for chunk in iter_keysight_chunks(reference_files, channel=channel, chunk_size=chunk_size):
    # iter_keysight_chunks exposes standard-unit arrays in this project.
    chunk_time_ns = np.asarray(chunk["time_ns"], dtype=float)
    chunk_voltage_mV = np.asarray(chunk["voltage_mV"], dtype=float)
    chunk_time_ns = chunk_time_ns - chunk_time_ns[0]
    if time_ns is None:
        time_ns = chunk_time_ns
    elif len(time_ns) != len(chunk_time_ns) or not np.allclose(time_ns, chunk_time_ns):
        raise ValueError(f"Time axis changed in {chunk['filename']}")
    if max_events is not None:
        remaining = max_events - events_loaded
        if remaining <= 0:
            break
        chunk_voltage_mV = chunk_voltage_mV[:remaining]
    waveform_parts.append(chunk_voltage_mV)
    events_loaded += len(chunk_voltage_mV)

waveforms_mV = np.concatenate(waveform_parts, axis=0)
dt_ns = float(np.median(np.diff(time_ns)))
print(f"Loaded {len(waveforms_mV):,} waveforms from {len(reference_files)} files")
print(f"Shape: {waveforms_mV.shape}; dt={dt_ns:.5g} ns; duration={time_ns[-1]-time_ns[0]:.3g} ns")

## Iterative median template and pulse-contamination test

In [ ]:
def robust_rms(values, axis=1):
    center = np.median(values, axis=axis, keepdims=True)
    return 1.4826 * np.median(np.abs(values - center), axis=axis)

smooth_samples = max(1, int(round(pulse_test_smoothing_ns / dt_ns)))
clean_mask = np.ones(len(waveforms_mV), dtype=bool)
for iteration in range(template_iterations):
    baseline_template_mV = np.median(waveforms_mV[clean_mask], axis=0)
    residuals_mV = waveforms_mV - baseline_template_mV
    # Remove only an event-specific scalar offset; preserve waveform structure.
    residuals_mV -= np.median(residuals_mV, axis=1, keepdims=True)
    event_robust_rms_mV = robust_rms(residuals_mV)
    smoothed_mV = uniform_filter1d(residuals_mV, size=smooth_samples, axis=1, mode="nearest")
    negative_excursion_mV = -np.min(smoothed_mV, axis=1)
    pulse_test_statistic = np.divide(
        negative_excursion_mV, event_robust_rms_mV,
        out=np.full(len(residuals_mV), np.inf), where=event_robust_rms_mV > 0,
    )
    clean_mask = pulse_test_statistic < pulse_test_snr
    print(f"Iteration {iteration + 1}: {clean_mask.sum():,}/{len(clean_mask):,} ({clean_mask.mean():.2%}) reference traces retained")

if clean_mask.mean() < minimum_clean_fraction:
    raise RuntimeError(
        f"Only {clean_mask.mean():.1%} of traces passed. Inspect the plots and "
        "adjust pulse_test_smoothing_ns/pulse_test_snr before saving a template."
    )

clean_waveforms_mV = waveforms_mV[clean_mask]
baseline_template_mV = np.median(clean_waveforms_mV, axis=0)
mean_waveform_mV = np.mean(clean_waveforms_mV, axis=0)
residuals_mV = clean_waveforms_mV - baseline_template_mV
residuals_mV -= np.median(residuals_mV, axis=1, keepdims=True)
residual_sigma_mV = 1.4826 * np.median(np.abs(residuals_mV), axis=0)
residual_q001_mV, residual_q01_mV, residual_q99_mV, residual_q999_mV = np.quantile(
    residuals_mV, [0.001, 0.01, 0.99, 0.999], axis=0
)

## Template and contamination diagnostics

In [ ]:
rng = np.random.default_rng(random_seed)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sample = rng.choice(len(waveforms_mV), size=min(30, len(waveforms_mV)), replace=False)
axes[0, 0].plot(time_ns, waveforms_mV[sample].T, color="0.6", alpha=0.18, lw=0.7)
axes[0, 0].plot(time_ns, baseline_template_mV, color="tab:blue", lw=2, label="Clean pointwise median")
axes[0, 0].plot(time_ns, mean_waveform_mV, color="tab:orange", lw=1.2, label="Clean mean")
axes[0, 0].set(xlabel="Time [ns]", ylabel="Voltage [mV]", title="Raw references and templates")
axes[0, 0].legend()

axes[0, 1].hist(pulse_test_statistic, bins=150, log=True)
axes[0, 1].axvline(pulse_test_snr, color="tab:red", label=f"Contamination threshold = {pulse_test_snr:g}")
axes[0, 1].set(xlabel="Smoothed negative excursion / robust RMS", ylabel="Events", title="Likely pulse contamination")
axes[0, 1].legend()

axes[1, 0].fill_between(time_ns, residual_q001_mV, residual_q999_mV, alpha=0.22, label="0.1–99.9%")
axes[1, 0].fill_between(time_ns, residual_q01_mV, residual_q99_mV, alpha=0.35, label="1–99%")
axes[1, 0].plot(time_ns, residual_sigma_mV, lw=1, label="Robust sigma")
axes[1, 0].set(xlabel="Time [ns]", ylabel="Residual [mV]", title="Empirical residual-noise envelope")
axes[1, 0].legend()

axes[1, 1].hist(event_robust_rms_mV[clean_mask], bins=100, alpha=0.8, label="Retained references")
axes[1, 1].set(xlabel="Per-event robust RMS [mV]", ylabel="Events", title="Residual noise scale")
axes[1, 1].legend()
fig.tight_layout()

In [ ]:
flagged = np.flatnonzero(~clean_mask)
if len(flagged):
    show = rng.choice(flagged, size=min(20, len(flagged)), replace=False)
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.plot(time_ns, (waveforms_mV[show] - baseline_template_mV).T, alpha=0.35, lw=0.8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set(xlabel="Time [ns]", ylabel="Template-subtracted voltage [mV]", title=f"Likely pulse-contaminated references (showing {len(show)} of {len(flagged)})")
    fig.tight_layout()
else:
    print("No reference traces were flagged as pulse-contaminated.")

## Empirical hardware-trigger recommendation

This calculation uses the raw negative excursion relative to the global baseline center, so coherent pickup remains included just as it would be seen by the oscilloscope trigger. A recommendation is trustworthy only when the reference exposure contains enough windows to measure the requested tail probability.

In [ ]:
window_duration_s = (time_ns[-1] - time_ns[0] + dt_ns) * 1e-9
reference_exposure_s = len(clean_waveforms_mV) * window_duration_s
global_baseline_mV = float(np.median(baseline_template_mV))
hardware_negative_excursion_mV = global_baseline_mV - np.min(clean_waveforms_mV, axis=1)

trigger_rows = []
for target_rate_hz in target_false_trigger_rates_hz:
    probability_per_window = 1.0 - np.exp(-target_rate_hz * window_duration_s)
    expected_exceedances = len(clean_waveforms_mV) * probability_per_window
    noise_excursion_quantile_mV = float(np.quantile(
        hardware_negative_excursion_mV, 1.0 - probability_per_window
    ))
    recommended_excursion_mV = noise_excursion_quantile_mV + trigger_safety_margin_mV
    trigger_rows.append({
        "target_false_rate_hz": target_rate_hz,
        "probability_per_window": probability_per_window,
        "expected_tail_events": expected_exceedances,
        "noise_excursion_quantile_mV": noise_excursion_quantile_mV,
        "safety_margin_mV": trigger_safety_margin_mV,
        "recommended_negative_excursion_mV": recommended_excursion_mV,
        "recommended_scope_level_mV": global_baseline_mV - recommended_excursion_mV,
        "supported_by_exposure": expected_exceedances >= minimum_expected_exceedances,
    })
trigger_recommendations = pd.DataFrame(trigger_rows)
print(f"Clean reference exposure: {reference_exposure_s:.6g} s in {len(clean_waveforms_mV):,} windows")
print(f"Global baseline level: {global_baseline_mV:.3f} mV")
display(trigger_recommendations)
if not trigger_recommendations["supported_by_exposure"].all():
    print("Warning: unsupported rows probe a rarer tail than this reference exposure can measure reliably. Collect more random-trigger windows before using them.")

sorted_excursion_mV = np.sort(hardware_negative_excursion_mV)
exceedance_counts = np.arange(len(sorted_excursion_mV), 0, -1)
empirical_false_rate_hz = exceedance_counts / reference_exposure_s
fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(sorted_excursion_mV, empirical_false_rate_hz, label="Empirical noise-trigger rate")
for row in trigger_rows:
    if row["supported_by_exposure"]:
        ax.scatter(row["recommended_negative_excursion_mV"], row["target_false_rate_hz"], s=45)
ax.set(xlabel="Negative trigger excursion below baseline [mV]", ylabel="Estimated accidental rate [Hz]", title="Noise-only trigger threshold from LED-off references")
ax.legend()
fig.tight_layout()

## Frequency-domain diagnostic

In [ ]:
fs_hz = 1e9 / dt_ns
psd_sample = residuals_mV[:min(2000, len(residuals_mV))]
frequency_hz, psd = welch(psd_sample, fs=fs_hz, axis=1, nperseg=min(2048, residuals_mV.shape[1]))
median_psd = np.median(psd, axis=0)
fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(frequency_hz / 1e6, median_psd)
ax.set(xlabel="Frequency [MHz]", ylabel="Median PSD [mV²/Hz]", title="Residual frequency content after median-template subtraction")
fig.tight_layout()

## Save reusable reference artifact

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
metadata = {
    "voltage_V": reference_voltage_V,
    "reference_collection": reference_collection,
    "channel": channel,
    "reference_data_dir": str(reference_data_dir),
    "reference_files": [str(path) for path in reference_files],
    "events_loaded": int(len(waveforms_mV)),
    "events_retained": int(clean_mask.sum()),
    "clean_fraction": float(clean_mask.mean()),
    "dt_ns": dt_ns,
    "pulse_test_smoothing_ns": pulse_test_smoothing_ns,
    "pulse_test_snr": pulse_test_snr,
    "reference_exposure_s": reference_exposure_s,
    "global_baseline_mV": global_baseline_mV,
}
np.savez_compressed(
    template_output,
    time_ns=time_ns,
    baseline_template_mV=baseline_template_mV,
    mean_waveform_mV=mean_waveform_mV,
    residual_sigma_mV=residual_sigma_mV,
    residual_q001_mV=residual_q001_mV,
    residual_q01_mV=residual_q01_mV,
    residual_q99_mV=residual_q99_mV,
    residual_q999_mV=residual_q999_mV,
    hardware_negative_excursion_mV=hardware_negative_excursion_mV,
    trigger_target_false_rate_hz=trigger_recommendations["target_false_rate_hz"].to_numpy(),
    trigger_recommended_scope_level_mV=trigger_recommendations["recommended_scope_level_mV"].to_numpy(),
    trigger_supported_by_exposure=trigger_recommendations["supported_by_exposure"].to_numpy(),
    metadata_json=json.dumps(metadata),
)
print(f"Saved baseline reference to {template_output.resolve()}")
display(metadata)